In [ ]:
import cdsapi
import os

# Initialize CDS API
c = cdsapi.Client()

# Dictionary of selected countries (where value == 1)
# and their approximate bounding boxes: [North, West, South, East]
countries = {
    "austria": [49.0, 9.5, 46.3, 17.2],
    "belgium": [51.5, 2.5, 49.4, 6.5],
    "bosnia_and_herz": [45.3, 15.7, 42.5, 19.7],
    "croatia": [46.6, 13.4, 42.3, 19.5],
    "finland": [70.1, 20.5, 59.8, 31.6],
    "greece": [41.8, 19.3, 34.8, 28.3],
    "hungary": [48.6, 16.1, 45.7, 22.9],
    "latvia": [58.1, 20.9, 55.6, 28.3],
    "lithuania": [56.5, 20.9, 53.8, 26.9],
    "poland": [54.9, 14.1, 49.0, 24.2],
    "portugal": [42.2, -9.6, 36.9, -6.1],
    "romania": [48.3, 20.2, 43.6, 29.8],
    "serbia": [46.2, 18.8, 42.2, 23.1],
    "slovakia": [49.6, 16.8, 47.7, 22.6],
    "spain": [43.8, -9.4, 35.9, 4.4],
    "switzerland": [47.9, 5.9, 45.8, 10.5],
    "ukraine": [52.4, 22.1, 44.3, 40.3],
}

# Years and months to download
years = range(2019, 2026)  # 2019 to 2025
months = [f"{m:02d}" for m in range(1, 13)]

# Iterate through each country and its bounding box
for country, area in countries.items():
    # Create a dedicated directory for each country
    output_dir = f"./era5/{country}/"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n--- Processing {country.upper()} ---")

    # Download NetCDF files month by month
    for year in years:
        for month in months:
            # Clean, dynamic file naming
            filename = f"{output_dir}era5_{country}_{year}_{month}.nc"

            if not os.path.exists(filename):  # skip if already downloaded
                print(f"Downloading {country} {year}-{month} ...")
                try:
                    c.retrieve(
                        "reanalysis-era5-single-levels",
                        {
                            "product_type": "reanalysis",
                            "variable": ["2m_temperature"],
                            "year": str(year),
                            "month": month,
                            "day": [f"{d:02d}" for d in range(1, 32)],
                            "time": [f"{h:02d}:00" for h in range(24)],  # hourly
                            "area": area,
                            "format": "netcdf",
                        },
                        filename,
                    )
                except Exception as e:
                    print(f"Failed to download {country} {year}-{month}: {e}")
            else:
                print(f"Skipped {country} {year}-{month} (file already exists)")

In [ ]:
import glob
import xarray as xr
import os

countries = [
    "austria",
    "belgium",
    "bosnia_and_herz",
    "croatia",
    "finland",
    "greece",
    "hungary",
    "latvia",
    "lithuania",
    "poland",
    "portugal",
    "romania",
    "serbia",
    "slovakia",
    "spain",
    "switzerland",
    "ukraine",
]

for country in countries:
    print(f"\nProcessing {country.upper()}...")

    file_pattern = f"./era5/{country}/era5_{country}_*.nc"
    files = sorted(glob.glob(file_pattern))

    if not files:
        print(f"  No files found for {country}. Skipping.")
        continue

    try:
        # Open all downloaded files as a single xarray dataset
        ds = xr.open_mfdataset(files, combine="by_coords", engine="netcdf4")

        # Convert 2m temperature from Kelvin to Celsius
        temp_c = ds["t2m"] - 273.15

        # --------------------------------------------------
        # 1. Calculate and Save Mean Temperature
        # --------------------------------------------------
        temp_avg = temp_c.mean(dim=["latitude", "longitude"])
        df_mean = temp_avg.to_dataframe(name="temp_mean").reset_index()
        df_mean = df_mean[["time", "temp_mean"]]  # Keep only relevant columns
        df_mean.rename(columns={"time": "datetime"}, inplace=True)

        csv_mean = f"{country}_hourly_temperature_2019_2025.csv"
        df_mean.to_csv(csv_mean, index=False)
        print(f"  Mean CSV saved as '{csv_mean}'")

        # --------------------------------------------------
        # 2. Calculate and Save Temperature Volatility (Std Dev)
        # --------------------------------------------------
        temp_volatility = temp_c.std(dim=["latitude", "longitude"])
        df_volatility = temp_volatility.to_dataframe(name="temp_std").reset_index()
        df_volatility = df_volatility[["time", "temp_std"]]  # Keep only relevant columns
        df_volatility.rename(columns={"time": "datetime"}, inplace=True)

        csv_volatility = f"{country}_hourly_temperature_volatility_2019_2025.csv"
        df_volatility.to_csv(csv_volatility, index=False)
        print(f"  Volatility CSV saved as '{csv_volatility}'")

        # Close the dataset to free up memory
        ds.close()

    except Exception as e:
        print(f"  Error processing {country}: {e}")

print("\nAll available countries have been processed!")

In [ ]:
import cdsapi
import os

# Initialize CDS API
c = cdsapi.Client()

# Dictionary of selected countries and their bounding boxes: [North, West, South, East]
countries = {
    "austria": [49.0, 9.5, 46.3, 17.2],
    "belgium": [51.5, 2.5, 49.4, 6.5],
    "bosnia_and_herz": [45.3, 15.7, 42.5, 19.7],
    "croatia": [46.6, 13.4, 42.3, 19.5],
    "finland": [70.1, 20.5, 59.8, 31.6],
    "greece": [41.8, 19.3, 34.8, 28.3],
    "hungary": [48.61, 16.0765, 45.7, 22.9],
    "latvia": [58.1, 20.9, 55.6, 28.3],
    "lithuania": [56.5, 20.9, 53.8, 26.9],
    "poland": [54.9, 14.1, 49.0, 24.2],
    "portugal": [42.2, -9.6, 36.9, -6.1],
    "romania": [48.3, 20.2, 43.6, 29.8],
    "serbia": [46.2, 18.8, 42.2, 23.1],
    "slovakia": [49.6, 16.8, 47.7, 22.6],
    "spain": [43.8, -9.4, 35.9, 4.4],
    "switzerland": [47.9, 5.9, 45.8, 10.5],
    "ukraine": [52.4, 22.1, 44.3, 40.3],
}

# Years and months to download (2019 through 2025)
years = range(2019, 2026)
months = [f"{m:02d}" for m in range(1, 13)]

# Iterate through each country
for country, area in countries.items():
    # Create a dedicated directory for each country's SSRD data
    output_dir = f"./era5_ssrd/{country}/"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n--- Processing SSRD for {country.upper()} ---")

    # Download NetCDF files month by month
    for year in years:
        for month in months:
            filename = f"{output_dir}era5_ssrd_{country}_{year}_{month}.nc"

            # Skip if already downloaded
            if os.path.exists(filename):
                print(f"Skipped {country} {year}-{month} (file already exists)")
                continue

            print(f"Downloading SSRD for {country} {year}-{month} ...")

            try:
                # Retrieve ERA5 single-level surface solar radiation downwards
                c.retrieve(
                    "reanalysis-era5-single-levels",
                    {
                        "product_type": "reanalysis",
                        "variable": ["surface_solar_radiation_downwards"],
                        "year": str(year),
                        "month": month,
                        "day": [f"{d:02d}" for d in range(1, 32)],
                        "time": [f"{h:02d}:00" for h in range(24)],
                        "area": area,
                        "format": "netcdf",
                    },
                    filename,
                )
            except Exception as e:
                print(f"Failed to download SSRD for {country} {year}-{month}: {e}")

print("\n✅ All SSRD downloads completed.")

In [ ]:
import glob
import xarray as xr
import os

# List of the 17 processed countries
countries = [
    "austria",
    "belgium",
    "bosnia_and_herz",
    "croatia",
    "finland",
    "greece",
    "hungary",
    "latvia",
    "lithuania",
    "poland",
    "portugal",
    "romania",
    "serbia",
    "slovakia",
    "spain",
    "switzerland",
    "ukraine",
]

for country in countries:
    print(f"\nProcessing SSRD for {country.upper()}...")

    # Locate all downloaded SSRD files for this country
    file_pattern = f"./era5_ssrd/{country}/era5_ssrd_{country}_*.nc"
    files = sorted(glob.glob(file_pattern))

    if not files:
        print(f"  No files found for {country}. Skipping.")
        continue

    try:
        # Open all downloaded files as a single xarray dataset
        ds = xr.open_mfdataset(files, combine="by_coords", engine="netcdf4")

        # Extract the SSRD variable
        # Note: ERA5 natively stores SSRD in Joules per square meter (J/m²).
        ssrd_data = ds["ssrd"]

        # --------------------------------------------------
        # 1. Calculate and Save Mean SSRD
        # --------------------------------------------------
        ssrd_avg = ssrd_data.mean(dim=["latitude", "longitude"])
        df_mean = ssrd_avg.to_dataframe(name="ssrd_mean").reset_index()
        df_mean = df_mean[["time", "ssrd_mean"]]  # Keep only relevant columns
        df_mean.rename(columns={"time": "datetime"}, inplace=True)

        csv_mean = f"{country}_hourly_ssrd_2019_2025.csv"
        df_mean.to_csv(csv_mean, index=False)
        print(f"  Mean SSRD CSV saved as '{csv_mean}'")

        # --------------------------------------------------
        # 2. Calculate and Save SSRD Volatility (Std Dev)
        # --------------------------------------------------
        ssrd_volatility = ssrd_data.std(dim=["latitude", "longitude"])
        df_volatility = ssrd_volatility.to_dataframe(name="ssrd_std").reset_index()
        df_volatility = df_volatility[["time", "ssrd_std"]]  # Keep only relevant columns
        df_volatility.rename(columns={"time": "datetime"}, inplace=True)

        csv_volatility = f"{country}_hourly_ssrd_volatility_2019_2025.csv"
        df_volatility.to_csv(csv_volatility, index=False)
        print(f"  Volatility SSRD CSV saved as '{csv_volatility}'")

        # Close the dataset to free up memory
        ds.close()

    except Exception as e:
        print(f"  Error processing {country}: {e}")

print("\nAll available SSRD countries have been processed!")

In [ ]:
import cdsapi
import os

# Initialize CDS API
c = cdsapi.Client()

# Dictionary of selected countries and their bounding boxes: [North, West, South, East]
countries = {
    "austria": [49.0, 9.5, 46.3, 17.2],
    "belgium": [51.5, 2.5, 49.4, 6.5],
    "bosnia_and_herz": [45.3, 15.7, 42.5, 19.7],
    "croatia": [46.6, 13.4, 42.3, 19.5],
    "finland": [70.1, 20.5, 59.8, 31.6],
    "greece": [41.8, 19.3, 34.8, 28.3],
    "hungary": [48.61, 16.0765, 45.7, 22.9],
    "latvia": [58.1, 20.9, 55.6, 28.3],
    "lithuania": [56.5, 20.9, 53.8, 26.9],
    "poland": [54.9, 14.1, 49.0, 24.2],
    "portugal": [42.2, -9.6, 36.9, -6.1],
    "romania": [48.3, 20.2, 43.6, 29.8],
    "serbia": [46.2, 18.8, 42.2, 23.1],
    "slovakia": [49.6, 16.8, 47.7, 22.6],
    "spain": [43.8, -9.4, 35.9, 4.4],
    "switzerland": [47.9, 5.9, 45.8, 10.5],
    "ukraine": [52.4, 22.1, 44.3, 40.3],
}

# Years and months to download (2019 through 2025)
years = range(2019, 2026)
months = [f"{m:02d}" for m in range(1, 13)]

# Variables needed to calculate wind speed and direction at 10m and 100m
wind_variables = ["10m_u_component_of_wind", "10m_v_component_of_wind", "100m_u_component_of_wind", "100m_v_component_of_wind"]

# Iterate through each country
for country, area in countries.items():
    output_dir = f"./era5_wind/{country}/"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n--- Downloading WIND data for {country.upper()} ---")

    for year in years:
        for month in months:
            filename = f"{output_dir}era5_wind_{country}_{year}_{month}.nc"

            if os.path.exists(filename):
                print(f"Skipped {country} {year}-{month} (file already exists)")
                continue

            print(f"Downloading WIND for {country} {year}-{month} ...")

            try:
                c.retrieve(
                    "reanalysis-era5-single-levels",
                    {
                        "product_type": "reanalysis",
                        "variable": wind_variables,
                        "year": str(year),
                        "month": month,
                        "day": [f"{d:02d}" for d in range(1, 32)],
                        "time": [f"{h:02d}:00" for h in range(24)],
                        "area": area,
                        "format": "netcdf",
                    },
                    filename,
                )
            except Exception as e:
                print(f"Failed to download WIND for {country} {year}-{month}: {e}")

print("\n✅ All WIND downloads completed.")

In [ ]:
import glob
import xarray as xr
import numpy as np
import pandas as pd
import os

countries = [
    "austria",
    "belgium",
    "bosnia_and_herz",
    "croatia",
    "finland",
    "greece",
    "hungary",
    "latvia",
    "lithuania",
    "poland",
    "portugal",
    "romania",
    "serbia",
    "slovakia",
    "spain",
    "switzerland",
    "ukraine",
]

wind_levels = {"10m": ("u10", "v10"), "100m": ("u100", "v100")}

for country in countries:
    print(f"\nProcessing WIND for {country.upper()}...")

    # 1. Load all downloaded wind NetCDFs for the current country
    file_pattern = f"./era5_wind/{country}/era5_wind_{country}_*.nc"
    files = sorted(glob.glob(file_pattern))

    if not files:
        print(f"  No files found for {country}. Skipping.")
        continue

    try:
        ds = xr.open_mfdataset(files, combine="by_coords", engine="netcdf4")

        # 2. Compute wind speed and direction
        for level, (u_name, v_name) in wind_levels.items():
            u = ds[u_name]
            v = ds[v_name]

            # Wind speed (m/s)
            ds[f"{level}_wind_speed"] = np.sqrt(u**2 + v**2)

            # Wind direction (meteorological convention: degrees from north)
            # Note: You compute this here, but the saving logic below only saves wind speed.
            ds[f"{level}_wind_direction"] = (180 / np.pi) * np.arctan2(-u, -v)

        # 3. Average and std over country area
        for level in ["10m", "100m"]:
            mean_speed = ds[f"{level}_wind_speed"].mean(dim=["latitude", "longitude"])
            std_speed = ds[f"{level}_wind_speed"].std(dim=["latitude", "longitude"])

            # Convert to DataFrames
            df_mean_speed = mean_speed.to_dataframe(name=f"{level}_wind_speed_mean").reset_index()
            df_std_speed = std_speed.to_dataframe(name=f"{level}_wind_speed_std").reset_index()

            # Handle time coordinate safely and clean up columns
            for df in [df_mean_speed, df_std_speed]:
                if "valid_time" in df.columns:
                    df.rename(columns={"valid_time": "datetime"}, inplace=True)
                elif "time" in df.columns:
                    df.rename(columns={"time": "datetime"}, inplace=True)

            # Keep only the datetime and the target value columns
            df_mean_speed = df_mean_speed[["datetime", f"{level}_wind_speed_mean"]]
            df_std_speed = df_std_speed[["datetime", f"{level}_wind_speed_std"]]

            # Save mean and std CSVs
            mean_csv = f"{country}_hourly_{level}_wind_mean_2019_2025.csv"
            std_csv = f"{country}_hourly_{level}_wind_std_2019_2025.csv"

            df_mean_speed.to_csv(mean_csv, index=False)
            df_std_speed.to_csv(std_csv, index=False)

            print(f"  ✅ Saved mean and std for {level} wind.")

        ds.close()

    except Exception as e:
        print(f"  Error processing {country}: {e}")

print("\n🎉 All wind data processed successfully.")